# 06 - Direct / unrolled learned inversion

Lecture section: 4.5, 4.7  |  Spine term this tutorial changes: the prior $R$ (and the solver
itself) is **learned end-to-end** for one specific operator $A$

$$\hat{x} = \arg\min_x\ \underbrace{D(Ax, y)}_{\text{data fidelity}} + \underbrace{R(x)}_{\text{prior, learned by unrolling}}$$

There are two routes to a *learned* prior. Notebook 5 (PnP) plugged a **generic, pretrained**
denoiser into the solver -- it works for *any* operator $A$. Here we take the other route:
**unroll** a fixed number of solver iterations into a neural network and **train it end-to-end**
on $(x, y)$ pairs for **one** physics $A$. The denoiser and the per-step parameters are learned
jointly. (We use 64 px and a tiny dataset so the loop trains in seconds on CPU -- enough to *show
the mechanism*; real results need much more data and training.)

In [1]:
import tutorial_common as tc
import deepinv as dinv
import torch
from deepinv.optim.data_fidelity import L2
from deepinv.optim.prior import PnP
from torch.utils.data import DataLoader

tc.set_seed()
N = 64  # smaller than the rest of the lecture so the training loop runs in seconds on CPU

deepinv 0.4.1 | torch 2.9.1 | device cpu


## A trainable unrolled network around the fixed physics $A$

We unroll $K=4$ proximal-gradient steps. The prior is a small CNN denoiser (DnCNN); the per-step
stepsizes and denoiser strengths are trainable. `unfold=True` turns the optimizer into a trainable
network. Crucially, the network is **built around this specific $A$** -- it is specialised to it.

In [2]:
phys = tc.ct_physics(angles=40, sigma=0.02, size=N)   # the SAME sparse-view CT, fixed
K = 4                                                  # number of unrolled layers (solver steps)
denoiser = dinv.models.DnCNN(in_channels=1, out_channels=1, depth=7, pretrained=None)

model = dinv.optim.PGD(
    stepsize=[1.0] * K,
    sigma_denoiser=[0.03] * K,
    trainable_params=["stepsize", "sigma_denoiser"],   # learn the per-layer parameters...
    data_fidelity=L2(),
    prior=PnP(denoiser=denoiser),                      # ...and the denoiser inside the prior
    max_iter=K,
    unfold=True,
).to(tc.DEVICE)
n_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_param}")

trainable parameters: 185865


## Before training, the (randomly initialised) network is useless

In [3]:
x_test = tc.load_hero(N)
y_test = phys(x_test)
model.eval()
with torch.no_grad():
    rec_untrained = model(y_test, phys)
print("untrained PSNR:", round(tc.psnr(rec_untrained, x_test), 2))

untrained PSNR: 10.69


## Train end-to-end (supervised) on a handful of random phantoms

The loss is simply $\|\text{network}(y) - x\|^2$. Measurements are generated **online** from the
same physics $A$. This explicit loop is what `deepinv.Trainer` does at scale.

In [4]:
trainset = dinv.utils.RandomPhantomDataset(size=N, length=24)   # varied phantoms (B,1,N,N)
loader = DataLoader(trainset, batch_size=4, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
loss_hist = []
for epoch in range(8):
    for xb in loader:
        xb = xb.to(tc.DEVICE)            # clean phantoms
        yb = phys(xb)                    # online noisy measurements y = A x + noise
        rec = model(yb, phys)            # forward through the unrolled network
        loss = ((rec - xb) ** 2).mean()  # supervised MSE
        opt.zero_grad()
        loss.backward()
        opt.step()
        loss_hist.append(loss.item())
print(f"trained {len(loss_hist)} steps; final loss {loss_hist[-1]:.4f}")

trained 48 steps; final loss 0.0018


## After training, the same network reconstructs an unseen image

In [5]:
model.eval()
with torch.no_grad():
    rec_trained = model(y_test, phys)
print("trained PSNR:", round(tc.psnr(rec_trained, x_test), 2))

tc.save_images(
    [x_test, rec_untrained, rec_trained],
    titles=[
        "x (ground truth)",
        tc.title_psnr("untrained net", rec_untrained, x_test),
        tc.title_psnr("trained unrolled net", rec_trained, x_test),
    ],
    fname="06_unrolled.png",
    suptitle="Unrolled network trained end-to-end for this operator A",
)
tc.save_curves(
    {"supervised MSE": loss_hist},
    fname="06_loss.png",
    xlabel="training step", ylabel="MSE loss", title="End-to-end training of the unrolled net",
)

trained PSNR: 16.74


saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/06_unrolled.png


saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/06_loss.png


## Unrolled vs Plug-and-Play: specialised vs general

- **Unrolled (here):** the network is **trained for this specific $A$**. Change $A$ (more/fewer
  angles, another modality) and you must retrain. In exchange, with enough data it can be fast (a
  fixed, small number of layers) and very accurate.
- **PnP (notebook 5):** a generic pretrained denoiser is dropped into the solver and works for
  **any** $A$ with no retraining -- more flexible, but it doesn't exploit a training set tuned to $A$.

Same spine either way -- data-fidelity $D$ fixed, prior $R$ learned. The difference is *how much*
of the solver is learned, and whether it is tied to one operator.

## Takeaway

Unrolling turns the iterative solver itself into a trainable network, specialised end-to-end to
one forward operator $A$. Powerful when you have training data and a fixed $A$ -- but
operator-specific, unlike the plug-and-play generality of notebook 5.